# 01 — Reproduction of CAD-ViT for Diabetic Retinopathy Severity Classification

**Research question.** How much of paper-guided CAD-ViT grading can the implemented candidate reproduce on frozen APTOS2019 and DDR protocols? This study has three generations: V1, V2 with Ben Graham σ=10, and balanced / DDR recovery. Validation chooses checkpoints; frozen final test is retrospective evidence only.

**Start here.** [Paper DOI](https://doi.org/10.1109/ACCESS.2025.3572455) · [historical 105-cell archive](../reproduction/archive/CAD_ViT_original_105_cells.ipynb) · [V1](../reproduction/experiments/01_experiment_initial_cadvit_v1.ipynb) · [V2](../reproduction/experiments/02_experiment_bg10_cadvit_v2.ipynb) · [balanced/recovery](../reproduction/experiments/03_experiment_balanced_training_and_ddr_recovery.ipynb) · [audit](../docs/RESEARCH_CONSOLIDATION_AUDIT.md).

The archive remains the immutable record of historical code and stored output. The reusable candidate in `reproduction/cadvit/` is an auditable implementation, **not proof that every historical run used identical code**. All new execution must write outside Git.

## 1. Scope, datasets, and paper fidelity

APTOS2019 comes from the [Kaggle competition](https://www.kaggle.com/competitions/aptos2019-blindness-detection/data); image redistribution is not assumed. DDR comes from the [OIA-DDR project](https://github.com/nkicsl/DDR-dataset). Use only authorized images and the original frozen manifests. No dataset download or split generation is performed here.

The paper reports a CAD-ViT architecture of about **7.26M parameters**. The current five-grade candidate is about **5.28M**; coordinate attention, dilated fusion, head design, preprocessing, and training choices include implementation assumptions. V2's Ben Graham background operation (σ=10) is an additional research choice, not a paper requirement. Consequently, this is a **partial, paper-guided reproduction**, not architectural equivalence. See the [source model](../reproduction/cadvit/model.py) and [methodology audit](../docs/RESEARCH_CONSOLIDATION_AUDIT.md) for the actual graph and evidence.

## 2. Environment and safe paths

The next cell detects the repository and checks optional dependencies without installing or downloading anything. Set `DR_RESEARCH_ROOT`, `DR_ARTIFACT_ROOT`, and `DR_OUTPUT_ROOT` before running on Colab or HPC. `DR_OUTPUT_ROOT` must be external to the repository and separate from historical runs. All action switches default to `False`; rerunning this cell is safe.

In [ ]:
from pathlib import Path
import os, sys, importlib.util, hashlib, json, csv
from IPython.display import display, Markdown

def show_table(rows, columns):
    if not rows:
        print("No recorded rows available")
        return
    header = "| " + " | ".join(columns) + " |"
    rule = "| " + " | ".join("---" for _ in columns) + " |"
    values = []
    for row in rows:
        values.append("| " + " | ".join(f"{row.get(k, '—'):.4f}" if isinstance(row.get(k), float)
                                      else str(row.get(k, '—')) for k in columns) + " |")
    display(Markdown("\n".join([header, rule, *values])))

def find_root():
    configured = os.environ.get("DR_RESEARCH_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / "README.md").is_file():
            raise FileNotFoundError(f"DR_RESEARCH_ROOT is not a repository: {root}")
        return root
    for root in (Path.cwd(), *Path.cwd().parents):
        if (root / "README.md").is_file() and (root / "novelty_integrated" / "e5").is_dir():
            return root.resolve()
    raise FileNotFoundError("Set DR_RESEARCH_ROOT to the cloned repository")

ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_ROOT = Path(os.environ.get("DR_ARTIFACT_ROOT", ROOT.parent / "Kel_13")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("DR_OUTPUT_ROOT", ROOT.parent / "research_new_runs")).expanduser().resolve()

def verified_example(dataset, image_path, split):
    """Verify a display image is exactly a frozen train/validation manifest row."""
    if split not in {"train", "val"}:
        raise ValueError("Only train/val examples are permitted")
    manifest = ARTIFACT_ROOT / "cad_vit_runs" / "manifests" / f"{dataset}_seed42.csv"
    if not manifest.is_file() or not manifest.with_suffix(".json").is_file():
        return False
    if hashlib.sha256(manifest.read_bytes()).hexdigest() != MANIFEST_SHA256[dataset]:
        raise ValueError("Frozen manifest hash mismatch")
    target = Path(image_path).expanduser().resolve()
    root = (ARTIFACT_ROOT / "datasets" / dataset).resolve()
    with manifest.open(encoding="utf-8", newline="") as stream:
        for row in csv.DictReader(stream):
            if row["split"] == "test":
                continue  # do not inspect test path, ID, label or hash
            if row["split"] != split:
                continue
            candidate = (root / row["image_path"]).resolve()
            if root not in candidate.parents:
                raise ValueError("Manifest path escapes dataset root")
            if candidate == target:
                if hashlib.sha256(target.read_bytes()).hexdigest() != row["sha256"]:
                    raise ValueError("Example image hash mismatch")
                print("Verified frozen", dataset, split, "example ID:", row["image_id"])
                return row["image_id"]
    raise ValueError("Example is not a frozen train/validation row; test access refused")
RUN_PREFLIGHT = False
RUN_MODEL_SANITY = False
RUN_SMOKE = False
RUN_TRAINING = False
RUN_RESUME = False
print("repository:", ROOT)
print("external artifacts:", ARTIFACT_ROOT, "available:", ARTIFACT_ROOT.is_dir())
print("external output (no directory created):", OUTPUT_ROOT)
print("Python:", sys.version.split()[0])
print({name: importlib.util.find_spec(name) is not None for name in ("torch", "torchvision", "transformers", "PIL", "cv2", "numpy", "pandas", "matplotlib")})
if importlib.util.find_spec("torch"):
    import torch
    print("torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
else:
    print("SKIPPED_MISSING_DATA: PyTorch-dependent model cells require the pinned project environment")

### Optional Colab storage

The next cell only mounts Google Drive when explicitly enabled. Set its external data and output paths in environment variables, then rerun setup. It does not silently clone a repository or install packages.

In [ ]:
# Run only when using Colab and after obtaining authorized dataset access.
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Set DR_ARTIFACT_ROOT and DR_OUTPUT_ROOT, then rerun the setup cell")
else:
    print("Drive mount skipped; no packages or weights downloaded")

### Frozen manifest identity

The next cell checks the known APTOS2019 and DDR manifest CSV hashes and sidecar presence **when files exist**. It does not open images or test rows. Missing external data is reported truthfully. The training runner performs stronger group/image checks before any run.

In [ ]:
from pathlib import Path
MANIFEST_SHA256 = {"APTOS2019": "73d1546e3c2fd2bf9e3d0fd10af914cb523d28e4bc8e1f25212af454d4be8854",
                   "DDR": "7bc3e3916e012bc46fcd9cd69aa469e0daf3229e1273f1d8ecb6054899bf6eb1"}
for dataset, expected in MANIFEST_SHA256.items():
    csv_path = ARTIFACT_ROOT / "cad_vit_runs" / "manifests" / f"{dataset}_seed42.csv"
    sidecar = csv_path.with_suffix(".json")
    if not csv_path.is_file() or not sidecar.is_file():
        print(dataset, "SKIPPED_MISSING_DATA: frozen CSV/sidecar not present on this host")
        continue
    actual = hashlib.sha256(csv_path.read_bytes()).hexdigest()
    assert actual == expected, f"{dataset} manifest SHA mismatch"
    print(dataset, "frozen CSV SHA-256 verified; sidecar present:", sidecar)
    print("For training, use the source runner's full image/group/hash preflight")

## 3. Exploratory class support from recorded validation evidence

This cell summarizes class support already stored in the **final-test aggregate** only to describe imbalance retrospectively; it never influences model selection. Training-specific counts require the frozen manifest. The plot is computed from committed JSON, contains no fundus image, and is safe to rerun.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("SKIPPED_MISSING_DATA: matplotlib unavailable; committed figures remain visible")
result_paths = {
    "APTOS balanced": ROOT / "results/APTOS2019/full_oversampling_amp_fp16_80epochs/final_test_summary.json",
    "DDR recovery": ROOT / "results/DDR/full_oversampling_fp32_math_mb8_80epochs/final_test_summary.json",
}
for label, path in result_paths.items():
    doc = json.loads(path.read_text(encoding="utf-8"))
    metrics = doc["metrics"] if "metrics" in doc else doc
    matrix = metrics.get("confusion_matrix")
    if matrix:
        support = [sum(row) for row in matrix]
        if plt is not None:
            plt.bar(range(5), support, alpha=.65, label=label)
        print(label, "retrospective test support:", support)
if plt is not None:
    plt.xlabel("DR grade"); plt.ylabel("Images"); plt.title("Recorded final-test class support")
    plt.legend(); plt.show()

## 4. Real-image preprocessing and training-only augmentation

`preprocess_fundus` performs EXIF-aware RGB conversion, crop, 512×512 resize, and Gaussian blur, returning a **uint8 image array**. `preprocess_fundus_v2` adds BG10 enhancement. In `experiments.dataset_for`, training augmentation follows preprocessing (horizontal/vertical flips and affine); tensor conversion occurs last. These choices belong to the candidate implementation. The next cell displays an authorized **train/validation** image only if you set `DR_SAMPLE_IMAGE` and `DR_SAMPLE_SPLIT`; it rejects a test-labeled sample and never writes the image. Missing inputs show `SKIPPED_MISSING_DATA`. Re-running is safe.

In [ ]:
sample = os.environ.get("DR_SAMPLE_IMAGE")
split = os.environ.get("DR_SAMPLE_SPLIT", "")
SAMPLE_VERIFIED = False
SAMPLE_ROW_ID = None
if not sample:
    print("SKIPPED_MISSING_DATA: set DR_SAMPLE_IMAGE to an authorized train/validation image")
elif split not in {"train", "val"}:
    print("SKIPPED_MISSING_DATA: set DR_SAMPLE_SPLIT=train or val; test images are excluded")
elif not (SAMPLE_ROW_ID := verified_example(os.environ.get("DR_SAMPLE_DATASET", "APTOS2019"), sample, split)):
    print("SKIPPED_MISSING_DATA: frozen manifest and sidecar unavailable for image verification")
else:
    SAMPLE_VERIFIED = True
    from PIL import Image
    import numpy as np
    from reproduction.cadvit.preprocessing import preprocess_fundus, preprocess_fundus_v2
    with Image.open(sample) as image:
        raw = np.asarray(image.convert("RGB"))
        v1 = preprocess_fundus(image)
        v2 = preprocess_fundus_v2(image)
    if plt is not None:
        fig, ax = plt.subplots(1, 3, figsize=(12, 4))
        for axis, pixels, label in zip(ax, (raw, v1, v2), ("raw", "V1 candidate", "V2 BG10")):
            axis.imshow(pixels); axis.set_title(label); axis.axis("off")
        plt.tight_layout(); plt.show()
    else:
        display(Image.fromarray(v1), Image.fromarray(v2))
    print("Training-only transforms are applied by experiments.dataset_for after preprocessing")

## 5. Model, objective, and implementation sanity check

The [model source](../reproduction/cadvit/model.py) defines the candidate graph and [experiments.py](../reproduction/cadvit/experiments.py) defines datasets, cross-entropy training, validation, checkpoint selection, and guarded final-test execution. The paper's approximate count is a reference, not an equality check. The next cell builds a **new untrained CPU model only when enabled**, counts actual parameters, and performs a 64×64 synthetic forward without data or checkpoint access. It does not load weights. Safe to rerun if CPU memory permits.

The following graph summarizes the **candidate source path**, not an exact reconstruction of the paper diagram. Module presence and parameter count are checked against `model.py` and the synthetic source test.

```mermaid
flowchart LR
    I[512×512 RGB tensor] --> S[Convolutional stem]
    S --> R[Interleaved inverted residual blocks]
    R --> A[Four coordinate-attention IRBs]
    A --> E[Three EnhancedMobileViT blocks with multiscale fusion]
    E --> H[Convolutional head and global pooling]
    H --> C[Five-grade linear classifier]
```

In [ ]:
if not RUN_MODEL_SANITY:
    print("SKIPPED: set RUN_MODEL_SANITY=True for untrained CPU parameter/synthetic-output check")
else:
    from reproduction.cadvit.experiments import VARIANTS
    from reproduction.cadvit.model import CADViTCandidate
    import torch
    variant = VARIANTS["aptos_r16"]
    candidate = CADViTCandidate(variant.model_config()).cpu().eval()
    print("Actual candidate parameters:", sum(p.numel() for p in candidate.parameters()))
    with torch.inference_mode():
        logits = candidate(torch.zeros(1, 3, 64, 64))
    assert logits.shape == (1, 5) and torch.isfinite(logits).all()
    print("Synthetic output shape:", tuple(logits.shape))

## 6. Campaign identity and training protocol

V1 and V2 historical details remain in their linked reports. Balanced APTOS uses its own 80-epoch campaign; DDR recovery comprises epoch 1–42 and 43–80 under **different numerical protocols** after instability. Training, validation, and final-test numbers must not be merged. The cell below reads all 80 recorded epochs for each recent campaign and plots loss and validation macro-F1; it does not retrain. Header and field names come from the actual JSONL rows.

In [ ]:
campaigns = {
 "APTOS balanced": ROOT / "results/APTOS2019/full_oversampling_amp_fp16_80epochs/history.jsonl",
 "DDR recovery": ROOT / "results/DDR/full_oversampling_fp32_math_mb8_80epochs/history.jsonl",
}
for label, path in campaigns.items():
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    assert len(rows) == 80 and [r["epoch"] for r in rows] == list(range(1, 81))
    print(label, "80 original epochs; keys:", list(rows[0]))
    print("first/last epoch:", rows[0]["epoch"], rows[-1]["epoch"])
    show_table(rows, ["epoch", "train_loss", "val_loss", "val_accuracy", "val_macro_f1"])

**APTOS2019 balanced: complete original 80-epoch validation table from `results/APTOS2019/full_oversampling_amp_fp16_80epochs/history.jsonl`.**

| Epoch | Train loss | Validation loss | Validation accuracy | Validation macro-F1 |
| --- | --- | --- | --- | --- |
| 1 | 1.300262 | 1.044935 | 0.554645 | 0.362002 |
| 2 | 1.136483 | 0.893349 | 0.655738 | 0.488097 |
| 3 | 1.003351 | 0.826451 | 0.685792 | 0.535251 |
| 4 | 0.914154 | 0.736216 | 0.724044 | 0.571386 |
| 5 | 0.867898 | 0.635216 | 0.745902 | 0.585542 |
| 6 | 0.829913 | 0.699582 | 0.724044 | 0.548546 |
| 7 | 0.831880 | 0.732517 | 0.707650 | 0.559662 |
| 8 | 0.807091 | 0.696550 | 0.737705 | 0.587706 |
| 9 | 0.795767 | 0.712637 | 0.721311 | 0.591522 |
| 10 | 0.823811 | 0.649131 | 0.751366 | 0.624572 |
| 11 | 0.792641 | 0.751202 | 0.713115 | 0.577371 |
| 12 | 0.747556 | 0.690402 | 0.707650 | 0.583081 |
| 13 | 0.759120 | 0.781316 | 0.683060 | 0.560761 |
| 14 | 0.744483 | 0.675961 | 0.756831 | 0.603070 |
| 15 | 0.729476 | 0.637292 | 0.784153 | 0.642580 |
| 16 | 0.742072 | 0.737432 | 0.718579 | 0.563451 |
| 17 | 0.692405 | 0.710679 | 0.751366 | 0.619353 |
| 18 | 0.705312 | 0.709754 | 0.734973 | 0.617880 |
| 19 | 0.689060 | 0.717440 | 0.737705 | 0.606333 |
| 20 | 0.684146 | 0.755579 | 0.721311 | 0.579415 |
| 21 | 0.687278 | 0.687112 | 0.762295 | 0.624651 |
| 22 | 0.654615 | 0.796990 | 0.718579 | 0.583938 |
| 23 | 0.669872 | 0.732279 | 0.767760 | 0.636601 |
| 24 | 0.642143 | 0.674393 | 0.792350 | 0.661421 |
| 25 | 0.664077 | 0.654506 | 0.781421 | 0.656728 |
| 26 | 0.658552 | 0.676960 | 0.767760 | 0.623384 |
| 27 | 0.643107 | 0.656150 | 0.762295 | 0.621732 |
| 28 | 0.628943 | 0.749213 | 0.732240 | 0.570349 |
| 29 | 0.666459 | 0.723543 | 0.759563 | 0.631464 |
| 30 | 0.638240 | 0.709785 | 0.759563 | 0.618390 |
| 31 | 0.647770 | 0.696690 | 0.737705 | 0.561977 |
| 32 | 0.617989 | 0.720003 | 0.740437 | 0.603769 |
| 33 | 0.597950 | 0.750939 | 0.745902 | 0.589645 |
| 34 | 0.572920 | 0.717154 | 0.756831 | 0.615568 |
| 35 | 0.547142 | 0.738748 | 0.745902 | 0.598820 |
| 36 | 0.555442 | 0.799296 | 0.751366 | 0.620236 |
| 37 | 0.536860 | 0.846372 | 0.759563 | 0.610461 |
| 38 | 0.530475 | 0.733562 | 0.770492 | 0.607041 |
| 39 | 0.516270 | 0.776527 | 0.748634 | 0.600354 |
| 40 | 0.499786 | 0.815584 | 0.759563 | 0.586109 |
| 41 | 0.488654 | 0.765832 | 0.759563 | 0.602373 |
| 42 | 0.497761 | 0.780163 | 0.748634 | 0.604181 |
| 43 | 0.487681 | 0.827277 | 0.726776 | 0.560234 |
| 44 | 0.464704 | 0.670800 | 0.775956 | 0.619953 |
| 45 | 0.476053 | 0.733724 | 0.773224 | 0.633475 |
| 46 | 0.469695 | 0.905131 | 0.721311 | 0.580118 |
| 47 | 0.469324 | 0.869881 | 0.743169 | 0.599039 |
| 48 | 0.433401 | 0.853997 | 0.770492 | 0.597404 |
| 49 | 0.434872 | 0.751238 | 0.770492 | 0.602464 |
| 50 | 0.427021 | 0.778591 | 0.778689 | 0.617779 |
| 51 | 0.422442 | 0.738813 | 0.775956 | 0.624863 |
| 52 | 0.407806 | 0.801775 | 0.740437 | 0.605539 |
| 53 | 0.423354 | 0.821797 | 0.756831 | 0.584939 |
| 54 | 0.419649 | 0.785133 | 0.770492 | 0.615659 |
| 55 | 0.399187 | 0.775003 | 0.792350 | 0.642133 |
| 56 | 0.397283 | 0.944823 | 0.767760 | 0.620803 |
| 57 | 0.369870 | 0.871357 | 0.754098 | 0.592674 |
| 58 | 0.366529 | 0.858352 | 0.765027 | 0.614404 |
| 59 | 0.378975 | 0.893307 | 0.784153 | 0.642383 |
| 60 | 0.372493 | 0.850395 | 0.743169 | 0.572085 |
| 61 | 0.356498 | 0.852188 | 0.729508 | 0.574338 |
| 62 | 0.348418 | 0.872986 | 0.775956 | 0.598962 |
| 63 | 0.385668 | 0.946846 | 0.721311 | 0.578648 |
| 64 | 0.349644 | 0.996220 | 0.751366 | 0.578105 |
| 65 | 0.360530 | 0.979983 | 0.745902 | 0.574987 |
| 66 | 0.346584 | 1.129620 | 0.724044 | 0.567427 |
| 67 | 0.373959 | 0.919862 | 0.778689 | 0.612870 |
| 68 | 0.360797 | 0.747373 | 0.756831 | 0.608913 |
| 69 | 0.373038 | 0.762192 | 0.751366 | 0.576954 |
| 70 | 0.344657 | 0.862721 | 0.756831 | 0.580989 |
| 71 | 0.348810 | 0.854702 | 0.797814 | 0.632991 |
| 72 | 0.320258 | 0.878562 | 0.778689 | 0.613173 |
| 73 | 0.323410 | 0.939972 | 0.759563 | 0.568303 |
| 74 | 0.320449 | 0.898563 | 0.770492 | 0.590485 |
| 75 | 0.332110 | 1.204536 | 0.721311 | 0.580409 |
| 76 | 0.333395 | 1.167203 | 0.762295 | 0.595723 |
| 77 | 0.312616 | 1.009468 | 0.770492 | 0.617550 |
| 78 | 0.308762 | 0.905324 | 0.797814 | 0.610958 |
| 79 | 0.312929 | 0.934144 | 0.778689 | 0.555060 |
| 80 | 0.311637 | 0.844686 | 0.784153 | 0.621923 |

**DDR recovery: complete original 80-epoch validation table from `results/DDR/full_oversampling_fp32_math_mb8_80epochs/history.jsonl`.** Epochs 1–42 and 43–80 use different numerical protocols.

| Epoch | Train loss | Validation loss | Validation accuracy | Validation macro-F1 |
| --- | --- | --- | --- | --- |
| 1 | 1.404246 | 1.225609 | 0.460863 | 0.340685 |
| 2 | 1.133719 | 1.109978 | 0.500799 | 0.387205 |
| 3 | 1.010772 | 1.022353 | 0.532748 | 0.422029 |
| 4 | 0.965450 | 0.968159 | 0.583866 | 0.442540 |
| 5 | 0.922552 | 1.069900 | 0.501597 | 0.443946 |
| 6 | 0.882990 | 0.888600 | 0.644569 | 0.507831 |
| 7 | 0.838678 | 1.074899 | 0.476038 | 0.459405 |
| 8 | 0.801186 | 1.011651 | 0.544728 | 0.502370 |
| 9 | 0.768221 | 0.984536 | 0.567891 | 0.480568 |
| 10 | 0.741517 | 0.873945 | 0.634984 | 0.524029 |
| 11 | 0.711949 | 0.914563 | 0.613419 | 0.513377 |
| 12 | 0.725335 | 0.964743 | 0.584665 | 0.507064 |
| 13 | 0.719023 | 0.866031 | 0.658147 | 0.535449 |
| 14 | 0.680094 | 0.813515 | 0.653355 | 0.568635 |
| 15 | 0.669315 | 0.991522 | 0.678115 | 0.537039 |
| 16 | 0.661969 | 0.984627 | 0.580671 | 0.513467 |
| 17 | 0.633065 | 0.923374 | 0.615016 | 0.558293 |
| 18 | 0.595616 | 0.920821 | 0.656550 | 0.560572 |
| 19 | 0.582316 | 0.812178 | 0.674121 | 0.563531 |
| 20 | 0.569431 | 0.775038 | 0.706869 | 0.556017 |
| 21 | 0.563140 | 0.840393 | 0.682907 | 0.549361 |
| 22 | 0.577795 | 0.897319 | 0.642173 | 0.532679 |
| 23 | 0.581485 | 0.901638 | 0.618211 | 0.536152 |
| 24 | 0.557117 | 1.529896 | 0.549521 | 0.417465 |
| 25 | 0.552339 | 0.955839 | 0.600639 | 0.517359 |
| 26 | 0.525822 | 0.852918 | 0.667732 | 0.560103 |
| 27 | 0.501575 | 0.674915 | 0.758786 | 0.608341 |
| 28 | 0.491974 | 0.793767 | 0.698882 | 0.577184 |
| 29 | 0.468824 | 0.915935 | 0.634984 | 0.549956 |
| 30 | 0.473203 | 1.042066 | 0.594249 | 0.523430 |
| 31 | 0.452893 | 0.758123 | 0.718051 | 0.598349 |
| 32 | 0.428039 | 0.723748 | 0.742812 | 0.620131 |
| 33 | 0.419914 | 0.704093 | 0.733227 | 0.583558 |
| 34 | 0.410195 | 0.880907 | 0.683706 | 0.575103 |
| 35 | 0.404584 | 0.728160 | 0.750000 | 0.595343 |
| 36 | 0.391914 | 0.859400 | 0.683706 | 0.566504 |
| 37 | 0.372283 | 0.809002 | 0.722843 | 0.551234 |
| 38 | 0.381347 | 0.720126 | 0.761981 | 0.611472 |
| 39 | 0.365142 | 0.802428 | 0.713259 | 0.589113 |
| 40 | 0.355888 | 0.775884 | 0.745208 | 0.589408 |
| 41 | 0.357476 | 0.817722 | 0.738818 | 0.615779 |
| 42 | 0.325936 | 0.803785 | 0.741214 | 0.612804 |
| 43 | 0.436829 | 0.800065 | 0.731629 | 0.611663 |
| 44 | 0.414366 | 0.711620 | 0.763578 | 0.612970 |
| 45 | 0.389670 | 0.747729 | 0.750799 | 0.631121 |
| 46 | 0.382753 | 0.786302 | 0.745208 | 0.598023 |
| 47 | 0.382461 | 0.693735 | 0.762780 | 0.608727 |
| 48 | 0.360790 | 0.699162 | 0.769968 | 0.629803 |
| 49 | 0.353551 | 0.746105 | 0.753195 | 0.610830 |
| 50 | 0.352060 | 0.708370 | 0.773962 | 0.627138 |
| 51 | 0.330419 | 0.692905 | 0.789936 | 0.646623 |
| 52 | 0.337432 | 0.667940 | 0.785942 | 0.621400 |
| 53 | 0.337916 | 0.710191 | 0.772364 | 0.642501 |
| 54 | 0.323587 | 0.714731 | 0.789936 | 0.647230 |
| 55 | 0.322200 | 1.030662 | 0.662939 | 0.563243 |
| 56 | 0.321953 | 0.944877 | 0.719649 | 0.611668 |
| 57 | 0.309921 | 0.798283 | 0.767572 | 0.624169 |
| 58 | 0.302576 | 0.668826 | 0.797923 | 0.632345 |
| 59 | 0.312219 | 0.807530 | 0.742812 | 0.606827 |
| 60 | 0.305733 | 0.931232 | 0.707668 | 0.592002 |
| 61 | 0.298884 | 0.954223 | 0.750799 | 0.622394 |
| 62 | 0.286900 | 0.814141 | 0.739617 | 0.609643 |
| 63 | 0.280343 | 0.929668 | 0.698882 | 0.617819 |
| 64 | 0.291308 | 0.724492 | 0.786741 | 0.634205 |
| 65 | 0.277409 | 0.774630 | 0.776358 | 0.622204 |
| 66 | 0.279468 | 0.940787 | 0.753195 | 0.614697 |
| 67 | 0.273348 | 0.733556 | 0.785144 | 0.639432 |
| 68 | 0.271504 | 0.877609 | 0.750799 | 0.641127 |
| 69 | 0.272224 | 0.790421 | 0.785942 | 0.653069 |
| 70 | 0.272320 | 0.808615 | 0.765176 | 0.628372 |
| 71 | 0.283457 | 0.725746 | 0.789936 | 0.643743 |
| 72 | 0.263404 | 0.724641 | 0.785942 | 0.620103 |
| 73 | 0.259554 | 0.738316 | 0.785942 | 0.631041 |
| 74 | 0.246140 | 0.763702 | 0.785942 | 0.647904 |
| 75 | 0.238237 | 0.900021 | 0.747604 | 0.625595 |
| 76 | 0.253734 | 0.768232 | 0.791534 | 0.633788 |
| 77 | 0.240036 | 0.782074 | 0.782748 | 0.647051 |
| 78 | 0.235022 | 0.844549 | 0.755591 | 0.617421 |
| 79 | 0.243834 | 0.988419 | 0.732428 | 0.631467 |
| 80 | 0.245128 | 0.833879 | 0.788339 | 0.626343 |

### Existing training curves

These are **stored figures from the recorded campaigns**, visible on GitHub without running a kernel. The APTOS curve and DDR recovery figure should be interpreted within their distinct protocols; the DDR transition after epoch 42 is a diagnostic recovery, not a single uninterrupted optimization protocol.

![APTOS balanced 80-epoch curves](../reproduction/report_figures/aptos_balanced_80_epoch_curves.svg)

![DDR recovery validation macro-F1](../results/DDR/full_oversampling_fp32_math_mb8_80epochs/validation_macro_f1.png)

![DDR recovery training loss](../results/DDR/full_oversampling_fp32_math_mb8_80epochs/train_loss.png)

## 7. Aggregate validation and final-test evaluation

The next cell loads only committed aggregate summaries. Validation macro-F1 selected checkpoints (APTOS epoch 24; DDR epoch 69). It prints **separate** validation and final-test metrics, per-grade rows, and actual confusion matrices. It does not load images or inference checkpoints. Final-test source files are linked in the historical reports.

In [ ]:
for label, folder in (("APTOS balanced", ROOT / "results/APTOS2019/full_oversampling_amp_fp16_80epochs"),
                      ("DDR recovery", ROOT / "results/DDR/full_oversampling_fp32_math_mb8_80epochs")):
    summary = json.loads((folder / "summary.json").read_text(encoding="utf-8"))
    final = json.loads((folder / "final_test_summary.json").read_text(encoding="utf-8"))
    metrics = final["metrics"] if "metrics" in final else final
    print("\n", label, "best validation epoch:", summary["best_epoch"],
          "validation macro-F1:", summary["best_validation_macro_f1"])
    print("final-test metrics:", {k: v for k, v in metrics.items() if isinstance(v, (float, int))})
    print("final-test per-grade:", metrics.get("per_class", metrics.get("per_grade", "see summary JSON")))
    print("final-test confusion matrix:", metrics.get("confusion_matrix"))

**Validation selection versus frozen final test, from the two distinct result summaries.**

| Dataset | Best val epoch | Val macro-F1 | Test accuracy | Test macro-F1 | Test weighted-F1 | Test QWK | Test grade MAE |
| --- | --- | --- | --- | --- | --- | --- | --- |
| APTOS2019 balanced | 24 | 0.661421 | 0.732240 | 0.531529 | 0.744946 | not reported | not reported |
| DDR recovery | 69 | 0.653069 | 0.778754 | 0.631981 | 0.801114 | 0.837079 | 0.305112 |

**Frozen final-test per-grade metrics (APTOS and DDR source labels retained).**

| Dataset | Grade | Support | Precision | Recall | F1 | Specificity |
| --- | --- | --- | --- | --- | --- | --- |
| APTOS2019 balanced | 0.000000 | 180 | 0.982558 | 0.938889 | 0.960227 | 0.983871 |
| APTOS2019 balanced | 1.000000 | 38 | 0.463415 | 0.500000 | 0.481013 | 0.932927 |
| APTOS2019 balanced | 2.000000 | 100 | 0.698925 | 0.650000 | 0.673575 | 0.894737 |
| APTOS2019 balanced | 3.000000 | 19 | 0.125000 | 0.210526 | 0.156863 | 0.919308 |
| APTOS2019 balanced | 4.000000 | 29 | 0.392857 | 0.379310 | 0.385965 | 0.949555 |
| DDR recovery | 0 | 626 | 0.907534 | 0.846645 | 0.876033 | 0.913738 |
| DDR recovery | 1 | 63 | 0.222892 | 0.587302 | 0.323144 | 0.891505 |
| DDR recovery | 2 | 448 | 0.862069 | 0.725446 | 0.787879 | 0.935323 |
| DDR recovery | 3 | 23 | 0.323529 | 0.478261 | 0.385965 | 0.981286 |
| DDR recovery | 4 | 92 | 0.791209 | 0.782609 | 0.786885 | 0.983621 |

### Paper comparison with metric definitions kept separate

The local [`PAPER_COMPARISONS`](../reproduction/cadvit/experiments.py) table transcribes paper-reported CAD-ViT **accuracy** of 88.6% on APTOS2019 and 84.8% on DDR, and paper-labeled **F1** of 88.4% and 84.4%, respectively. Our balanced/recovery values above are **macro-F1** on our frozen test splits. The paper F1 aggregation and split equivalence have not been independently established here; therefore the F1 columns must **not** be subtracted as like-for-like estimates. Our candidate's architecture and optimization also differ. This comparison is contextual, not evidence of exact reproduction.

### Historical evaluation figures and error interpretation

The original DDR final-test confusion matrix (true grades in rows, predictions in columns) has 1,252 samples and 975 correct predictions. Grade 3 has 23 samples, 11 correct. Grade 1 has the lowest per-class F1; grade 2 ↔ 3 confusion is visible, but no clinical cause is established from aggregate counts. For APTOS, consult its committed result JSON and source report for class-wise evidence.

![Original DDR final-test confusion matrix](../results/DDR/full_oversampling_fp32_math_mb8_80epochs/confusion_matrix.png)

## 8. Controlled *new-run* execution interface (disabled)

The original candidate runner is imported rather than copied. `matrix` lists configurations; `init` creates a fresh protocol (exclusive write), `preflight` checks source/data/protocol, and `train` requires `--execute-training`. Configure `DR_CADVIT_NEW_ARTIFACT_ROOT` to a **new external root** with frozen manifest CSV/sidecar copied unchanged into its expected `cad_vit_runs/manifests/` directory, and `DR_CADVIT_DATA_ROOT` to the authorized dataset directory. Initialize the fresh protocol once outside this notebook before enabling preflight. No historical run directory may be reused. It cannot recreate historical V1/V2 weights merely by rerunning current code. This notebook has no final-test execution cell.

In [ ]:
import subprocess
variant_name = "aptos_r16"  # change explicitly only after inspecting experiments.VARIANTS
new_root_text = os.environ.get("DR_CADVIT_NEW_ARTIFACT_ROOT", "")
new_root = Path(new_root_text).expanduser().resolve() if new_root_text else None
data_root_text = os.environ.get("DR_CADVIT_DATA_ROOT", "")
candidate_args = [sys.executable, "-m", "reproduction.cadvit.experiments",
                  "preflight", "--variant", variant_name, "--artifact-root", str(new_root or "<NEW_EXTERNAL_ROOT>")]
if data_root_text:
    candidate_args += ["--data-root", data_root_text]
print("Candidate preflight:", candidate_args)
if RUN_PREFLIGHT:
    if new_root is None or not new_root.is_dir() or ROOT in new_root.parents:
        print("SKIPPED_MISSING_DATA: new external candidate root and frozen manifests unavailable")
    else:
        subprocess.run(candidate_args, cwd=ROOT, check=True)
else:
    print("SKIPPED: RUN_PREFLIGHT=False")

### Explicit training gate

The next cell invokes the source `train` command only after **both** action flags are set and the new external root is configured. `experiments.py` owns optimizer, epoch reporting, validation macro-F1 selection, checkpoint saving and `--resume`; the notebook does not redefine them. Read the campaign configuration and run a source preflight first. Never aim `--artifact-root` at a historical run. Safe while flags remain false.

In [ ]:
train_args = [sys.executable, "-m", "reproduction.cadvit.experiments", "train",
              "--campaign", "paper_v1", "--variant", variant_name,
              "--artifact-root", str(new_root or "<NEW_EXTERNAL_ROOT>"), "--execute-training"]
if data_root_text:
    train_args += ["--data-root", data_root_text]
print("New candidate training command (review protocol before enabling):", train_args)
if RUN_TRAINING and RUN_PREFLIGHT:
    if new_root is None or not new_root.is_dir() or ROOT == new_root or ROOT in new_root.parents:
        raise RuntimeError("Training requires a fresh external artifact root")
    subprocess.run(train_args, cwd=ROOT, check=True)
else:
    print("SKIPPED: training disabled; no checkpoint or output changed")

## 9. Conclusion and limits

APTOS and DDR balanced/recovery training reached 80 epochs and have frozen final-test aggregate artifacts. V1/V2 historical results remain distinct. The candidate differs materially from the paper (~5.28M versus ~7.26M parameters); model graph, split, preprocessing, balancing, Gaussian enhancement and mixed DDR recovery constrain paper comparisons. Minority-grade support is small and no external clinical validation exists. This notebook supports retrospective inspection and guarded new candidate runs; exact historical reruns need their original binary checkpoints, manifests and environment.